# Example 3.7.2 / 1: Continue in a Second Session

Companion notebook for **3.7.2 Warm Starts and Snapshots** in the AMPL lab Documentation, section "Continue in a Second Session". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Continue in a Second Session

The run below solves the production model, takes a snapshot as a string, rebuilds the session in
a second `AMPL` object with `eval()`, and solves a changed scenario there:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                 # Products
param rate {PROD} > 0;                    # Production rate in tons per hour
param profit {PROD};                      # Profit per ton
param market {PROD} >= 0;                 # Maximum sales in tons
param avail >= 0;                         # Available production hours
var Make {p in PROD} >= 0, <= market[p];  # Tons to make, capped by sales
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # Total profit in dollars
subject to Time: sum {p in PROD} Make[p] / rate[p] <= avail;  # Hours used within available hours

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["market"] = {"bands": 6000, "coils": 4000}
ampl.param["avail"] = 40

In [5]:
# run.py
from amplpy import AMPL

ampl.solve(solver="highs", verbose=False)
assert ampl.solve_result == "solved"
saved = ampl.snapshot()

continued = AMPL()
continued.eval(saved)
print(continued.var["Make"].to_dict())
continued.param["avail"] = 45
continued.solve(solver="highs", verbose=False)
print(continued.solve_result, continued.obj["Total_Profit"].value())

{'bands': 6000, 'coils': 1400}
solved 213000.0
